# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [2]:
# imports
import os
from openai import OpenAI
from IPython.display import Markdown, display, update_display

In [4]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_OLLAMA='gemma3:4b'
OLLAMA_BASE_URL = "http://localhost:11434/v1"


In [5]:
# set up environment

In [7]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""
def getSystemPrompt():
    system_prompt="""
    You are an expert technical tutor designed to support learning during a course.

    Your role is to take any technical question related to:
    - Programming (especially Python)
    - Mathematics
    - Physics
    - Artificial Intelligence / Machine Learning
    - Or a mix of these topics

    And respond with a clear, structured, and learner-friendly explanation.

    Behavior rules:
    1. Adapt your explanation depth automatically based on the complexity of the question.
    2. Assume the user is learning the topic for the first time unless the question clearly implies advanced knowledge.
    3. Explain concepts step-by-step, starting from intuition before formal definitions.
    4. Use examples whenever helpful:
    - Python code for programming questions
    - Worked examples for math/physics
    - Simple models or analogies for AI/ML concepts
    5. When relevant, highlight common mistakes or misconceptions.
    6. Use clear formatting (headings, bullet points, numbered steps).
    7. Be precise, accurate, and educational — prioritize understanding over brevity.
    8. If a concept depends on prerequisites, briefly explain them instead of assuming prior knowledge.

    Do NOT:
    - Overwhelm the learner with unnecessary jargon
    - Skip reasoning steps
    - Assume unstated background knowledge

    Your goal is to help the learner truly understand the topic, not just get an answer.
    """
    return system_prompt

def getUserPrompt(question):
    user_prompt = f"""
    I am learning a technical subject as part of a course.

    Please explain the following question clearly and step-by-step, as if teaching it to a learner:

    {question}

    If helpful:
    - Include examples (Python code, math steps, or diagrams explained in text)
    - Explain the intuition before the formal solution
    - Point out common mistakes or misunderstandings
    """
    
    return user_prompt


In [9]:
# Get gpt-4o-mini to answer, with streaming

In [10]:
# Get Llama 3.2 to answer
openai = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


def getAnswer(question):
    messages = [
        {"role":"system","content":getSystemPrompt()},
        {"role":"user","content":getUserPrompt(question)}
    ]
    stream = openai.chat.completions.create(model = MODEL_OLLAMA,
                                              messages = messages,
                                              stream=True
                                              )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


In [11]:
getAnswer(question)

Okay, let's break down this Python code snippet step-by-step. It’s a concise way to extract authors from a list of book dictionaries, and it leverages a powerful Python feature called a generator expression combined with `yield from`.  We’ll build up our understanding gradually.

**1. The Big Picture – What Are We Trying To Do?**

Imagine you have a list of books, where each book is represented as a dictionary. Each dictionary has keys like "title," "author," "year," etc.  Our goal is to create a sequence (a kind of "stream") of *only* the authors' names. We want to do this efficiently, without creating a massive list of all the authors in memory at once.

**2. Let's Analyze the Parts**

Let's break down the code into its components, starting with the simplest:

*   **`books`**: We assume `books` is a variable that already holds a list of dictionaries. Each dictionary represents a book.  Let's represent a book like this:

    ```python
    books = [
        {"title": "The Lord of the Rings", "author": "J.R.R. Tolkien", "year": 1954},
        {"title": "Pride and Prejudice", "author": "Jane Austen", "year": 1813},
        {"title": "1984", "author": "George Orwell", "year": 1949},
        {},  # An empty book dictionary - important to consider!
        {"title": "To Kill a Mockingbird", "year": 1960} # Missing author
    ]
    ```

*   **`book.get("author")`**: The `.get()` method is used to safely retrieve a value from a dictionary.  It's better than directly accessing `book["author"]` because if the "author" key is *missing* from a book dictionary, `.get()` will return `None` (or a default value you specify - we won’t use a default here). Directly accessing a missing key with `book["author"]` would raise a `KeyError` and crash the program.

*   **`for book in books`**: This is a standard `for` loop. It iterates through each dictionary (`book`) in the `books` list.

*   **`if book.get("author")`**: This is the crucial filtering step. This condition checks if the author exists in the dictionary. The expression `book.get("author")` will return `None` if the "author" key is missing. In Python, a `None` value is considered "falsy" in a boolean context -- meaning it evaluates to `False`. Thus, this entire `if` statement effectively only processes books that *have* a value for the "author" key.

*   **`{... for ... in ... if ...}`**: This whole expression is called a *generator expression*. It’s a convenient way to create a sequence (like a list) of values based on a condition, but it does so *lazily*. This means it only calculates the values as they are needed, which can be much more efficient than creating a large list in memory.

*   **`yield from`**: This is the magic ingredient. `yield from` handles a nested generator expression. It effectively "flattens" the generator created by the expression.  It allows you to yield all the values generated by the inner generator expression one by one.

**3. Putting It All Together – The Step-by-Step Execution**

Let’s trace what happens when this code runs:

1.  The code starts with the `books` list (as defined above).
2.  The *generator expression* `{book.get("author") for book in books if book.get("author")}` is evaluated.
3.  The `for` loop iterates through each `book` in `books`.
4.  For each book, the `if` statement checks if `book.get("author")` returns a non-`None` value (i.e., if the book has an "author").
5.  Only if the author exists, `book.get("author")` is executed, and its value is yielded.
6.  The `yield from` effectively propagates these yielded values, creating a generator that produces the author names one at a time.

**4. The Resulting Generator**

The `yield from` statement creates a generator that yields the author names from `books` where the "author" key exists in the dictionary.  In our example, it will yield:

*   "J.R.R. Tolkien"
*   "Jane Austen"
*   "George Orwell"


**5. Common Mistakes and Misunderstandings**

*   **`KeyError`**:  The most common mistake is forgetting to use `.get()` and accidentally trying to access `book["author"]` directly. This will raise a `KeyError` if the "author" key is missing.
*   **Generator Expressions vs. Lists**:  It’s important to remember that this is a *generator expression*, not a list comprehension. Generator expressions are evaluated lazily (on demand), while list comprehensions create a whole list in memory at once. This difference in efficiency can be significant when dealing with large datasets.
*   **`yield from` Complexity**:  `yield from` can seem a bit confusing at first. It's essentially a way to "delegate" the yielding process from one generator to another, but understanding its role is crucial for working with more complex generator expressions.

**6. A Simple Example to Illustrate the Laziness**

If the `books` list were *huge*, the generator expression wouldn't create the entire list of authors in memory. Instead, it would generate the author names only when they are requested. This saves a lot of memory and time, especially when filtering large datasets.


Do you want to:

*   Walk through a specific example with a smaller `books` list?
*   Explore how this works with different types of data within the dictionaries?
*   Discuss how to handle default values if the "author" key is missing (instead of returning `None`)?